In [ ]:
import numpy as np
from collections import deque
import plotly.express as px
from sklearn.cluster import DBSCAN as DB

# Własny DBSCAN

class DBSCAN:
    def __init__(self, eps=1.0, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples
        self._clusters = None

    def fit(self, X: np.array):
        # Tworzenie grafów sąsiedztw
        neigh = [[] for _ in range(len(X))]
        visited = np.zeros(len(X))
        for i in range(len(X)):
            for j in range(i + 1, len(X)):
                dist = np.linalg.norm(X[i] - X[j])
                if dist <= self.eps:
                    neigh[i].append(j)
                    neigh[j].append(i)
         
        self._clusters = np.zeros(X.shape[0])
        clust_num = 0
        for i in range(len(neigh)):
            if visited[i]:
                continue

            # mark as possible noise and not visited
            if len(neigh[i]) + 1 < self.min_samples:
                self._clusters[i] = -1
                continue

            visited[i] = 1

            self._clusters[i] = clust_num
            q = deque(neigh[i])

            while q:
                v = q.popleft()
                if visited[v]:
                    continue

                visited[v] = 1
                
                # core point
                if len(neigh[v]) + 1 >= self.min_samples:
                    self._clusters[v] = clust_num
                    q.extend(neigh[v])        # expand cluster
                else:
                    self._clusters[v] = clust_num
                    
            clust_num += 1

        print(neigh)

In [52]:
from sklearn.datasets import make_blobs

noise = np.random.normal(0, 1, (10, 2))
X, _ = make_blobs(n_samples = 500,n_features = 2,centers = 5,random_state = 23)
        
X = np.vstack([X, noise])
print(X)
    

db = DBSCAN(eps=1.0, min_samples=4)
# X = np.array([[1, 1],[1, 0],[0, 1], [10, 0]])
db.fit(X)
db._clusters
print(db._clusters)

fig = px.scatter(x=X[:, 0], y=X[:, 1], color=db._clusters)

fig.show()

db_test = DB(eps=1.0, min_samples=4)
db_test.fit(X)
db_test.labels_

fig1 = px.scatter(x=X[:, 0], y=X[:, 1], color=db_test.labels_)

fig1.show()

print(np.allclose(db._clusters, db_test.labels_))

[[-5.98093403  4.67959779]
 [-4.7041755   2.91568022]
 [-0.476981    7.34114297]
 ...
 [ 0.40450937  0.99720087]
 [-0.87361203 -0.22228114]
 [ 0.46735198  0.43850766]]
[[32, 55, 82, 84, 91, 92, 145, 155, 174, 193, 266, 311, 319, 332, 357, 491, 494, 498], [12, 78, 79, 119, 125, 143, 176, 177, 209, 221, 224, 239, 247, 359, 383, 430, 462, 486], [3, 57, 68, 88, 111, 159, 202, 378, 455], [2, 24, 41, 60, 62, 88, 89, 111, 153, 159, 164, 198, 202, 214, 256, 263, 275, 287, 295, 338, 378, 400, 420, 455, 468, 469, 474, 479], [], [11, 28, 37, 48, 55, 84, 92, 107, 114, 146, 163, 171, 174, 180, 187, 193, 231, 239, 259, 265, 266, 276, 278, 285, 307, 315, 336, 364, 367, 391, 405, 425, 430, 440, 444, 454, 465, 476, 481, 484, 491], [34, 45, 136, 203, 227, 251], [16, 22, 67, 86, 97, 154, 170, 178, 206, 211, 232, 240, 248, 273, 317, 323, 334, 342, 360, 375, 386, 404, 409, 428, 472, 483, 492, 493], [9, 33, 58, 64, 81, 96, 113, 115, 123, 127, 140, 161, 182, 190, 200, 208, 217, 261, 288, 289, 292, 303, 309, 

True
